# 04 — ModelOpt Q/DQ ONNX Export

Two quantization paths are supported:

| Path | Backend | Best for | Supported dtypes |
|------|---------|----------|-----------------|
| **MOQ** (`modelopt.onnx`) | Post-export ONNX graph rewrite | `int8`, `fp8` | int8, fp8 |
| **MTQ** (`modelopt.torch`) | PyTorch-level fake-quant → ONNX export | `int4` (weight-only, covers Conv2d) | int4 |

> **Why two paths?** MOQ's INT4 mode only targets `MatMul`/`Gemm` (LLM-style).
> For CNNs like ResNet-18 where FLOPs live in `Conv2d`, MTQ inserts per-channel
> weight quantizers into every Conv/Linear, calibrates on real data, and emits
> proper Q/DQ nodes on ONNX export.

## 1 — Configuration

In [ ]:
QUANT_DTYPE   = "int4"                     # "int8", "fp8", or "int4"
ONNX_IN       = "../onnx/resnet18.onnx"    # float baseline
ONNX_OUT      = f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx"
CALIB_BATCHES = 32

## 2 — Common Setup (shared by both paths)

In [ ]:
from pathlib import Path
import sys, numpy as np, torch, onnx
from onnx import TensorProto

sys.path.insert(0, str(Path("../src").resolve()))

from data import build_runner_loaders
from config import ExperimentConfig
from model import get_model
from onnx_exporter import ONNXExporter

### 2a — Export float ONNX baseline (if it doesn't exist yet)

In [ ]:
if not Path(ONNX_IN).exists():
    _model = get_model(cfg=None, pretrained=False)
    ONNXExporter(
        model=_model, device="cpu", onnx_path=ONNX_IN,
    ).export_model(opset_version=17, dynamic_batch=True)
    print(f"Exported float baseline → {ONNX_IN}")
else:
    print(f"Float baseline already exists: {ONNX_IN}")

### 2b — Calibration helpers

In [ ]:
def build_numpy_calib_data(device="cpu", batch_size=32, n_batches=CALIB_BATCHES):
    """Return a single numpy array of calibration images (for MOQ)."""
    cfg = ExperimentConfig(device=device, batch_size=batch_size)
    loader, _ = build_runner_loaders(cfg)
    batches = []
    for i, (images, _) in enumerate(loader):
        if i >= n_batches:
            break
        batches.append(images.numpy())
    data = np.concatenate(batches, axis=0)
    print(f"Numpy calibration data: {data.shape}")
    return data


def build_torch_forward_loop(device="cpu", batch_size=32, n_batches=CALIB_BATCHES):
    """Return a forward-loop callable (for MTQ calibration)."""
    loader, _ = build_runner_loaders(ExperimentConfig(device=device, batch_size=batch_size))

    def forward_loop(model):
        model.eval()
        with torch.no_grad():
            for i, (images, _) in enumerate(loader):
                if i >= n_batches:
                    break
                model(images.to(device))

    return forward_loop

### 2c — ONNX inspection helper

In [ ]:
def inspect_qdq_graph(onnx_path):
    """Print Q/DQ counts and initializer dtype breakdown."""
    m = onnx.load(onnx_path)
    ops = [n.op_type for n in m.graph.node]
    q_count  = ops.count("QuantizeLinear")
    dq_count = ops.count("DequantizeLinear")
    print(f"Q: {q_count}, DQ: {dq_count}")

    dtype_map = {v: k for k, v in TensorProto.DataType.items()}
    counts = {}
    for init in m.graph.initializer:
        dtype = dtype_map.get(init.data_type, str(init.data_type))
        counts[dtype] = counts.get(dtype, 0) + 1
    print(f"Initializer dtypes: {counts}")

    # For int4, count weight tensors specifically
    n_int4 = sum(
        1 for init in m.graph.initializer
        if dtype_map.get(init.data_type, "") == "INT4" and "weight" in init.name
    )
    if n_int4:
        print(f"INT4 weight tensors: {n_int4}")

    return q_count, dq_count

## 3 — Quantize

The cell below dispatches to the right backend based on `QUANT_DTYPE`:

- **int8 / fp8** → `modelopt.onnx.quantization` (MOQ) — operates on the exported ONNX graph
- **int4** → `modelopt.torch.quantization` (MTQ) — inserts fake-quant in PyTorch, then re-exports to ONNX

### 3a — MOQ path (int8 / fp8)

In [ ]:
if QUANT_DTYPE in ("int8", "fp8"):
    import modelopt.onnx.quantization as moq

    calib_data = build_numpy_calib_data()

    # FP8 needs explicit node list (only Conv/Gemm/MatMul/Add)
    _nodes_to_q = None
    if QUANT_DTYPE == "fp8":
        import onnx as _onnx
        _FP8_OPS = {"Conv", "Gemm", "MatMul", "Add"}
        _m = _onnx.load(ONNX_IN)
        _nodes_to_q = [n.name for n in _m.graph.node if n.op_type in _FP8_OPS]
        print(f"FP8 explicit nodes_to_quantize: {len(_nodes_to_q)} "
              f"({sum(1 for n in _m.graph.node if n.op_type == 'Add')} Add nodes)")

    moq.quantize(
        onnx_path=ONNX_IN,
        quantize_mode=QUANT_DTYPE,
        calibration_data=calib_data,
        output_path=ONNX_OUT,
        op_types_to_quantize=list(_FP8_OPS) if QUANT_DTYPE == "fp8" else None,
        nodes_to_quantize=_nodes_to_q,
    )
    print(f"Saved → {ONNX_OUT}")
else:
    print(f"QUANT_DTYPE={QUANT_DTYPE!r} → skipping MOQ (handled by MTQ below)")

### 3b — MTQ path (int4 weight-only, covers Conv2d)

> The `modelopt.onnx` INT4 path only supports `MatMul`/`Gemm` (LLM-style
> weight-only quant). For ResNet-18, where ~all FLOPs live in `Conv2d`, that
> produces an effectively-FP32 model with only 1 INT4 layer.
>
> MTQ inserts weight quantizers into every `Conv2d`/`Linear` module, calibrates
> them on real data, and emits Q/DQ nodes on ONNX export — so the graph has
> INT4 weights for all the convs too.

In [ ]:
if QUANT_DTYPE == "int4":
    import modelopt.torch.quantization as mtq

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Fresh model for MTQ (it modifies in-place)
    torch_model = get_model(cfg=None, pretrained=False).to(device).eval()
    forward_loop = build_torch_forward_loop(device=device)

    # Per-output-channel INT4 weight-only quantization.
    # Targets every nn.Conv2d / nn.Linear weight tensor (axis=0 = per output-channel).
    # Activations stay FP32. BatchNorm is excluded (will fold into preceding Conv).
    INT4_CONV_CFG = {
        "quant_cfg": {
            "*weight_quantizer": {"num_bits": 4, "axis": 0, "enable": True},
            "*input_quantizer":  {"enable": False},
            "nn.BatchNorm1d":    {"*": {"enable": False}},
            "nn.BatchNorm2d":    {"*": {"enable": False}},
            "nn.BatchNorm3d":    {"*": {"enable": False}},
            "default":           {"enable": False},
        },
        "algorithm": "max",
    }

    mtq.quantize(torch_model, INT4_CONV_CFG, forward_loop)
    mtq.print_quant_summary(torch_model)

    # Export to ONNX
    torch_model.eval()
    dummy = torch.randn(1, 3, 224, 224, device=device)
    with torch.no_grad():
        torch.onnx.export(
            torch_model,
            dummy,
            ONNX_OUT,
            input_names=["images"],
            output_names=["logits"],
            opset_version=21,
            dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
            do_constant_folding=True,
        )
    print(f"Saved → {ONNX_OUT}")
else:
    print(f"QUANT_DTYPE={QUANT_DTYPE!r} → skipping MTQ (handled by MOQ above)")

## 4 — Inspect the quantized graph

In [ ]:
inspect_qdq_graph(ONNX_OUT)

## Notes

### Q/DQ counts by dtype

**INT4** — 0 Q, ~20 DQ (weight-only quantization)

Weights are pre-quantized to INT4 statically at export time and stored as INT4 constants in the graph. No Q node is needed at runtime — the quantization already happened. Only a DQ is inserted to convert INT4 weights → float just before the compute op. Activations stay in float entirely.

> With MOQ's ONNX path you'd see 0 Q / 1 DQ because it only targets `MatMul`/`Gemm`. With MTQ you get DQ nodes on every Conv2d too.

**INT8** — ~41 Q, ~41 DQ (full activation + weight quantization)

Both weights and activations are quantized dynamically. Every quantizable op gets a Q/DQ pair on inputs: ~20 Conv layers × 2 tensors (weight + activation) ≈ 40, plus the final FC layer = ~41.

**FP8** — ~38 Q, ~38 DQ (3 fewer than INT8)

Same scheme as INT8 (dynamic Q/DQ for both weights and activations), but fewer pairs. `modelopt`'s FP8 mode excludes certain layers that INT8 quantizes:
- The first Conv layer — input pixel distributions don't fit FP8's narrow range well
- Residual addition ops — FP8 arithmetic isn't supported for them in TensorRT
- The final FC/classifier — may be excluded for accuracy reasons

### Why INT4 is weight-only

INT4 has only 16 discrete values (or 8 for unsigned). That's far too coarse for activations, which vary dynamically at inference time. So instead of the full `→ Q → DQ → Op →` pattern, INT4 does:

```
INT4 weight (constant) → DQ → Conv/MatMul(activation_fp32, dequantized_weight)
```

There's no Q node because the quantization happened statically when the model was exported — the weights are already stored as INT4. You only need DQ to convert them back to float for computation.

### Why ResNet-18 has ~20 Conv Q/DQ pairs

ResNet18 has ~20 Conv layers. Each gets one Q/DQ for weights and one for activations (in INT8/FP8), giving ~40 pairs. You get 38–41 instead of exactly 40 because a few layers are typically excluded from quantization (e.g., the first conv or final FC layer, which are sensitive to precision loss).